# Setup

Before running this notebook, complete the steps below.

**1. Install** — see the [README](../README.md) (`uv sync` or `pip install .`).

**2. Download the MACE model**
```bash
mkdir -p ../models
wget https://github.com/ACEsuit/mace-foundations/releases/download/mace_mh_1/mace-mh-1.model \
     -O ../models/mace-mh-1.model
```

**3. Download paper data**: the dataset is on Zenodo at
[doi.org/10.5281/zenodo.19650620](https://doi.org/10.5281/zenodo.19650620)
(access-restricted — click **Request access** on the record page).
Download `cft_paper.zip` and extract it:
```bash
unzip cft_paper.zip -d /path/to/data/
```

**4. Fill in the paths** in the Configuration cell below and run it.
No shell exports or kernel restarts needed.

In [ ]:
import os

# ── Paths — edit these to match your local setup ──────────────────────────────
os.environ.setdefault('CFT_DATA_DIR',    '/path/to/cft_paper')        # extracted Zenodo archive
os.environ.setdefault('MODEL_PATH',      '../models/mace-mh-1.model') # MACE model file
os.environ.setdefault('CFT_SCRATCH_DIR', '.')                         # output dir for meshes / PNGs
os.environ.setdefault('MODEL_PATH_OMAT', '')                          # optional: OMAT model for relaxations

## init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# stdlib
import ast
import copy
import math
import itertools
from glob import glob
from pathlib import Path

# numerics / data
import numpy as np
import pandas as pd
from scipy.interpolate import make_interp_spline, griddata

# plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, MultipleLocator

# ASE
from ase import Atoms
from ase.build import fcc211
from ase.constraints import FixAtoms, FixedPlane, FixCartesian
from ase.geometry import find_mic
from ase.io import read, write, Trajectory
from ase.neb import NEB
from ase.optimize import BFGS
from ase.visualize import view

# autoadsorbate
from autoadsorbate import Fragment, Surface
from autoadsorbate.Particle import get_cube_surface_pts, grid_round_cube
from autoadsorbate.Surf import attach_fragment
from autoadsorbate.utils import get_blenderized

# cft
from cft import Manifold
from cft.mesh_interpolate import subdivide_ply_smooth_color
from cft.mesh_utils import (
    compute_outward_vertex_normals_quads,
    fit_line_and_distances,
    furthest_projected_pairs,
    get_manifold_minima,
    get_mesh_islands,
    points_close_to_rotating_line,
    values_to_colors,
    generate_plane_mesh
)
from cft.utils import parse_vec
from cft.plot_utils import add_linear_fits
from cft.neb_utils import ForceFit, fit_raw, fit_images, plot_band, get_neb_probe, plot_barrier, get_PMD_structure

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 9,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7
})


metal_colors = {
    'Ag': '#666666',   # darker gray
    'Au': '#F6D32D',   # brighter yellow
    'Cu': '#CC5500',   # slightly darker orange
    'Pd': '#003F88',   # darker blue
    'Pt': '#9BD3F5'    # much lighter blue
}

color_dict = {
    'e_particle_C': "#323333",
    'e_particle_O': "#910000",
    'e_C-O': "#fdfdfd",
    'sum_two_body': "#2d703b",
    'd_body': "#b1008a",
}

In [ ]:
from mace.calculators import mace_mp
clean_calc = mace_mp(
    model=os.environ.get('MODEL_PATH') or None,
    head='matpes_r2scan',
    device='cpu',
)


## Data

Set the following environment variables before running:

| Variable | Description |
|---|---|
| `CFT_DATA_DIR` | Path to the `cft_paper/` data directory (from Zenodo) |
| `MODEL_PATH` | Path to the MACE model file (`.model`) |
| `CFT_SCRATCH_DIR` | Directory for output files — meshes, trajectories, PNGs (default: `.`) |
| `MODEL_PATH_OMAT` | Path to MACE OMAT model for relaxations (optional, scratchpad only) |


In [ ]:
_cft_data = os.environ.get('CFT_DATA_DIR')
if not _cft_data:
    raise EnvironmentError(
        "CFT_DATA_DIR is not set. "
        "Run the Setup cell at the top of this notebook first."
    )
DATA_DIR = Path(_cft_data)
SCRATCH_DIR = Path(os.environ.get('CFT_SCRATCH_DIR', '.'))


### render assets

In [ ]:
particle = read(DATA_DIR / 'test_particle.xyz')
slab = read(DATA_DIR / 'test_slab.xyz')

### NEB seeds

In [ ]:
neb_seed_lst = read(DATA_DIR / 'neb_seed_lst.xyz', index=':')

### probes

In [ ]:
probes = [
    Fragment('ClC#[O+]', to_initialize=1),
    Fragment('Cl[O]', to_initialize=1),
    Fragment('Cl[C]', to_initialize=1),
    Fragment('ClO', to_initialize=1),
    Fragment('Cl[CH]', to_initialize=1),
    Fragment('Cl[CH2]', to_initialize=1),
    Fragment('ClC', to_initialize=1),
    Fragment('Cl[N]', to_initialize=1),
    Fragment('Cl[NH]', to_initialize=1),
    Fragment('ClN', to_initialize=1)
]


### isolated ref

In [ ]:
neb_images = read(DATA_DIR / 'images_neb.xyz', index=':')
PMD_image = neb_images[5].copy()
PMD_image = PMD_image[[atom.index for atom in PMD_image if atom.symbol in ['C','O']]]

In [ ]:
f_PMD_image =  get_neb_probe(PMD_image, keep=None)

In [ ]:
particle.calc = copy.deepcopy(clean_calc)
e_particle = particle.get_potential_energy()
PMD_image.calc = copy.deepcopy(clean_calc)
PMD_image.get_potential_energy()

PMD_CO = PMD_image[[atom.index for atom in PMD_image if atom.symbol in ['C', 'O']]]
PMD_CO.calc = copy.deepcopy(clean_calc)

O_atoms = Atoms(['O'], [[0,0,0]])
O_atoms.calc = copy.deepcopy(clean_calc)
C_atoms = Atoms(['C'], [[0,0,0]])
C_atoms.calc = copy.deepcopy(clean_calc)

isolated_ref = {
    'CO': np.sum([
        O_atoms.get_potential_energy(),
        C_atoms.get_potential_energy()
        - particle.get_potential_energy()
        ]
        ),
    'O': O_atoms.get_potential_energy(),
    'C': C_atoms.get_potential_energy(),
    'PMD_CO': PMD_CO.get_potential_energy()
}

In [ ]:
file_str = DATA_DIR / 'kuma_anisotropy_results/PMD*.xyz'
file_str = file_str.as_posix()
files = glob(file_str)

phi_grid_dict = {
    'CO': {},
    'O': {},
    'C': {},
}
for file in files:
    phi = int(file.split('phi')[-1].split('.xyz')[0])
    if '_COS_' in file:        
        phi_grid_dict['CO'][phi] = read(file)
        
    elif '_CS_' in file:
        phi_grid_dict['C'][phi] = read(file)
    elif '_OS_' in file:
        phi_grid_dict['O'][phi] = read(file)


In [ ]:
# Note: 'shuang_' prefix is a data-provenance naming artifact from the original dataset files.
cu_grid = read(DATA_DIR / 'metal_Cu_211_grid.xyz')
pd_grid = read(DATA_DIR / 'metal_Pd_211_grid.xyz')
au_grid = read(DATA_DIR / 'metal_Au_211_grid.xyz')

grids_dict = {
    'Cu': {
        'e_ClC#[O+]': cu_grid,
        'e_Cl[O]': cu_grid,
        'e_Cl': cu_grid 
    },
    'Pd': {
        'e_ClC#[O+]': pd_grid,
        'e_Cl[O]': pd_grid,
        'e_Cl': pd_grid 
    },
    'Au': {
        'e_ClC#[O+]': au_grid,
        'e_Cl[O]': au_grid,
        'e_Cl': au_grid 
    },
    'HEO': {
        'e_ClC#[O+]': read(DATA_DIR / 'shuang_PROBE-CO_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl': read(DATA_DIR / 'shuang_PROBE-H_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[O]': read(DATA_DIR / 'shuang_PROBE-O_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[CH2]': read(DATA_DIR / 'shuang_PROBECH2_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_ClC': read(DATA_DIR / 'shuang_PROBECH3_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[CH]': read(DATA_DIR / 'shuang_PROBECH_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[C]': read(DATA_DIR / 'shuang_PROBEC_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_ClN': read(DATA_DIR / 'shuang_PROBEH2N_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[NH]': read(DATA_DIR / 'shuang_PROBEHN_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_ClO': read(DATA_DIR / 'shuang_PROBEHO_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
        'e_Cl[N]': read(DATA_DIR / 'shuang_PROBEN_WRAPRatoms_PREC0.2_TSS2.5.xyz'),
    },
    'HEA': {
        'e_ClC#[O+]': read(DATA_DIR / '0_CO_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[O]': read(DATA_DIR / '1_O_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[C]': read(DATA_DIR / '2_C_WRAPRatoms_PREC0.3.xyz'),
        'e_ClO': read(DATA_DIR / '3_HO_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[CH]': read(DATA_DIR / '4_CH_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[CH2]': read(DATA_DIR / '5_CH2_WRAPRatoms_PREC0.3.xyz'),
        'e_ClC': read(DATA_DIR / '6_CH3_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[N]': read(DATA_DIR / '7_N_WRAPRatoms_PREC0.3.xyz'),
        'e_Cl[NH]': read(DATA_DIR / '8_HN_WRAPRatoms_PREC0.3.xyz'),
        'e_ClN': read(DATA_DIR / '9_H2N_WRAPRatoms_PREC0.3.xyz'),
    },
}

HEO_slab = read(DATA_DIR / 'Co0.2Ni0.2Mg0.2Zn0.2Mn0.2Al2O4COH_311_cond2.traj')
HEO_slab = HEO_slab[[atom.index for atom in HEO_slab if atom.symbol != 'X']]

slabs_dict = {
    'Cu': read(DATA_DIR / 'metal_Cu_211_slab.xyz'),
    'Pd': read(DATA_DIR / 'metal_Pd_211_slab.xyz'),
    'Au': read(DATA_DIR / 'metal_Au_211_slab.xyz'),
    'HEO': HEO_slab,
    'HEA': read(DATA_DIR / 'test_particle.xyz'),
}


slab_refs = {}
for metal, atoms in slabs_dict.items():
    _atoms = atoms.copy()
    _atoms.calc= copy.deepcopy(clean_calc)
    e_ref = _atoms.get_potential_energy()
    slab_refs[metal] = e_ref

probe_dict = {}

for p in probes:
    _atoms = p.get_conformer(0)[1:]
    _atoms.calc= copy.deepcopy(clean_calc)
    probe_dict['e_'+p.smile] = _atoms.get_potential_energy()

## Example usage

### Manifold fitting illustration

In [ ]:
touch_sphere_size=2.5
precision = 2.5

m = Manifold(particle, mode='particle', precision=precision, touch_sphere_size=touch_sphere_size)
# m.save_ply(filename=SCRATCH_DIR / 'low_poly_grid.ply')
xx = m.grid_atoms.copy()
# xx.symbols = np.array(['He' for _ in m.grid_atoms])
write(f'{SCRATCH_DIR}/atoms_hq_grid.xyz', xx)
# view(m.grid_atoms)

center = np.mean(m.atoms.positions, axis=0)

diffs = m.atoms.positions - center
dists = np.linalg.norm(diffs, axis=1)
index = np.argmax(dists)
particle_radius = dists[index]

grid_radius = particle_radius + touch_sphere_size + 0.5 # 0.5 is safety buffer

xyz, _, _ =get_cube_surface_pts(radius=grid_radius, center=center, d_min=precision)
xyz*=grid_radius
xyz+=center
cube_atoms = Atoms(['He' for _ in xyz], xyz)
m.grid = cube_atoms.positions
m.normals = compute_outward_vertex_normals_quads(m.grid, m.faces)
m.save_ply(filename=SCRATCH_DIR / 'low_poly_grid_cube.ply')


round_cube_geometry = grid_round_cube(center=center, radius=grid_radius, d_min=precision)[0]
round_cube_atoms = Atoms(['He' for _ in round_cube_geometry], round_cube_geometry)
m.grid = round_cube_atoms.positions
m.normals = - round_cube_atoms.positions + center
m.save_ply(filename=SCRATCH_DIR / 'low_poly_grid_round_cube.ply')


vectors = Atoms()
for i, v in enumerate(m.grid):
    for d in np.arange(0,2,.05):
        vectors += Atoms(['H'], [v+d*m.normals[i]])

grid_atoms = m.grid_atoms.copy()
grid_atoms.symbols = ['He' for _ in grid_atoms]


lst = [cube_atoms, round_cube_atoms, grid_atoms, vectors]

for i, a in enumerate(lst):
    write(f'{SCRATCH_DIR}/atoms_{i}.xyz', a)
    
# view(lst)

### Fragment to XYZ

In [ ]:
smils = [
    'Cl[C]',
    'ClC',
    'S1SC([O-])C1',
    'S1SN=[NH+]1',
    'Cl[OH+]CC(O)CO',
    'Cl[n+]1ccccc1',
]

f_trj = []

for i, a in enumerate([Fragment(smi, to_initialize=3, prune_rms_thresh=0).get_conformer(1) for smi in smils]):

    if a.symbols[0]=='S':

        a.positions -= a.positions[1]*.5
        a = a[2:]

        
    else:
        a = a[1:]

    for r in np.arange(0,1.5,0.05):
        a+=Atoms(['He'], [[0,0,r]])
    f_trj.append(a)
    write(f'{SCRATCH_DIR}/fragment_{i}.xyz', a)
view(f_trj)


### Manifolds with differnt touch_sphere_size (Morse curve figure)

In [ ]:
atoms = particle.copy()
view_atoms = atoms.copy()
precision = .5
touch_sphere_size=3.6

s = Surface(atoms, mode='particle', precision=precision, touch_sphere_size=touch_sphere_size)
sites_atoms = Atoms(['N' for _ in s.site_df.index.values],  [ v for v in s.site_df.coordinates.values])

center = np.mean(sites_atoms.positions, axis=0)
diffs = sites_atoms.positions - center
dists = np.linalg.norm(diffs, axis=1)
index = np.argmax(dists)
particle_radius = dists[index]
grid_radius = particle_radius + touch_sphere_size + 0.5 # 0.5 is safety buffer
round_cube_geometry = grid_round_cube(center=center, radius=grid_radius, d_min=precision)


site_container = Atoms()
for touch_sphere_size in np.arange(1.5, 3.6, .5):

    m = Manifold(sites_atoms, mode='particle', precision=precision, touch_sphere_size=touch_sphere_size, grid_mode = round_cube_geometry)

    for i in [4201,4187]: #top and bridge "site" hand picked
        site_container+=m.grid_atoms[i]

    print(f'{len(m.grid_atoms) = }, {touch_sphere_size = }')
    m.save_ply(filename=f'{SCRATCH_DIR}/grid_increments_{touch_sphere_size}.ply')
    view_atoms+=m.grid_atoms
# view(view_atoms[len(atoms):])
# view(view_atoms)


### neb in plane CO
**Paper figure: NEB energy decomposition (3-panel)**

In [ ]:
m = Manifold(particle.copy(), mode='particle', precision=.5, touch_sphere_size=2.5, wrap_on='sites')

In [ ]:
recepie = [
    [{'f' : Fragment('Cl[C]', to_initialize=1), 'site_i': 127},
    {'f' : Fragment('Cl[O]', to_initialize=1), 'site_i': 128}],
    [{'f' : Fragment('ClC#[O+]', to_initialize=1), 'site_i': 131}]
]

endpoint_trj = []
for i, ads_lst in enumerate(recepie):
    _atoms = m.atoms.copy()
    for dic in ads_lst:
        attach_fragment(
            atoms = _atoms,
            site_dict = m.site_df.iloc[dic['site_i']],
            fragment = dic['f'].get_conformer(0),
            n_rotation = 0,
            height = 1.5,
        ) 
    c1 = FixAtoms(indices=[atom.index for atom in _atoms if atom.symbol not in ['C', 'O']])
    # c2 = FixedPlane(
    #     indices=[atom.index for atom in atoms if atom.symbol in ['C', 'O']],
    #     direction=[1, 0, 0],
    # )

    indices = [atom.index for atom in _atoms if atom.symbol in ['C', 'O']]
    c2 = FixCartesian(indices, mask=(1, 0, 0))

    _atoms.set_constraint([c1,c2])
    _atoms.calc = copy.deepcopy(clean_calc)
    endpoint_trj.append(_atoms)

# view(endpoint_trj)            

In [ ]:
for atoms in endpoint_trj:
    traj = Trajectory(f'{SCRATCH_DIR}/trj_tmp.xyz', 'w')
    optimizer = BFGS(atoms, trajectory=traj, append_trajectory=True)
    optimizer.run(fmax = 0.01, steps=1000)

In [ ]:
images_restart = read(f'{DATA_DIR}/CO_neb/neb_CO_in_plane.xyz', index=':')
images_restart = images_restart[-11:]

In [ ]:
restart = True

if restart == False:
    initial = endpoint_trj[0].copy()
    final = endpoint_trj[1].copy()

    n_images = 11  
    images = [initial]
    for i in range(n_images - 2):
        image = initial.copy()
        images.append(image)
    images.append(final)

    neb = NEB(images, remove_rotation_and_translation=False)
    neb.interpolate(apply_constraint=True)

else:
    images = images_restart.copy()

neb = NEB(images, remove_rotation_and_translation=False)
for image in images:
    image.calc = copy.deepcopy(clean_calc)

traj = Trajectory(f'{SCRATCH_DIR}/neb_CO_in_plane_restart.xyz', 'w', images)

opt = BFGS(neb, trajectory=traj)
opt.run(fmax=0.1, steps=500)

In [ ]:
res = []
for i, image in enumerate(neb_images):
    e_neb_ref = image.get_potential_energy()
    info = {
            'image': i,
            'e_neb_ref':e_neb_ref
        }
    for k, v in {
        'C':['C'],
        'O':['O'],
        'CO':['C','O'],
        'particle':['Ag','Au','Cu','Pd','Pt'],
        'particle_C':['Ag','Au','Cu','Pd','Pt', 'C'],
        'particle_O':['Ag','Au','Cu','Pd','Pt', 'O'],
        }.items():
        
        a = image.copy()
        a = a[[atom.index for atom in a if atom.symbol in v]]
        a.calc = copy.deepcopy(clean_calc)
        e = a.get_potential_energy()
        info[f'e_{k}']= e
        
        if k == 'CO':
            info[f'd_{k}'] = a.get_all_distances()[0][1]
             
    res.append(info)

In [ ]:
a = read(f'../examples/CO_neb/particle_C_field_in_plane.xyz')
ref_C = a.arrays[f'e_ClC'].flatten()[0]
ref_C = ref_C - e_particle - isolated_ref['C']

a = read(f'../examples/CO_neb/particle_O_field_in_plane.xyz')
ref_O = a.arrays[f'e_ClO'].flatten()[0]
ref_O = ref_O - e_particle - isolated_ref['O']

In [ ]:
res_df = pd.DataFrame(res)

res_df['e_C-O'] = res_df['e_CO'] - isolated_ref['O'] - isolated_ref['C']
# res_df['e_C-O'] -= res_df['e_C-O'].values[0]
res_df['e_particle_O'] = res_df['e_particle_O'] - e_particle - isolated_ref['O']
res_df['e_particle_C'] = res_df['e_particle_C'] - e_particle - isolated_ref['C']
res_df['e_neb'] = res_df['e_neb_ref'] - e_particle - isolated_ref['C'] - isolated_ref['O']

res_df['sum_two_body'] = res_df['e_particle_O'] + res_df['e_particle_C'] + res_df['e_C-O']

res_df['d_body'] = res_df['e_neb'] - res_df['sum_two_body']


# res_df['dx'] = res_df['xx'] -res_df['e_neb']

res_df


In [ ]:
res_df['e_particle_O'].values[0] - res_df['e_particle_O'].values[-1]
res_df['e_particle_C'].values[0] - res_df['e_particle_C'].values[-1]
# res_df['e_C-O'].values[0] - res_df['e_C-O'].values[-1]

path = [0]
for i in range(len(neb_images) - 1):
    # positions = neb_images[0].positions
    dR = neb_images[i + 1].positions - neb_images[i].positions
    dR, _ = find_mic(dR, neb_images[i].cell, neb_images[i].pbc)
    path.append(path[i] + np.sqrt((dR**2).sum()))
res_df['path'] = path


In [ ]:
grd_CO= read(f'{DATA_DIR}/CO_neb/particle_CO_field_in_plane.xyz')

a = neb_images[-1].copy()
pos = a[[atom. index for atom in a if atom.symbol == 'C']].positions[0]
dr = np.linalg.norm(grd_CO.positions - pos, axis=1)

field_CO = grd_CO.arrays['e_ClC#O+']
field_CO = field_CO - isolated_ref['O'] - isolated_ref['C']

CO_at_star = field_CO[np.argmin(dr)]


In [ ]:
lw_pt = 0.2 * 72 / 25.4  # 0.2 mm in points

fig, axs = plt.subplots(
    1, 3,
    figsize=[182/25.4, 32.7*1.5/25.4],
    dpi=600
)

color_dict = {
    'e_particle_C': "#323333",
    'e_particle_O': "#910000",
    'e_C-O': "#fdfdfd",
    'sum_two_body': "#2d703b",
    'd_body': "#b1008a",
}

label_dict = {
    'e_particle_C': "particle + C",
    'e_particle_O': "particle + O",
    'e_C-O': "C + O",
    'sum_two_body': r"$\sum E_{\mathrm{two\ body}}$",
    'd_body': r"$E_{\mathrm{NEB}} - \sum E_{\mathrm{two\ body}}$",
}


ax = axs[0]

for k in ['e_particle_C', 'e_particle_O', 'e_C-O']:
    sns.scatterplot(
        x=res_df.path,
        y=res_df[k].values,
        label=label_dict[k],
        ax=ax,
        color=color_dict[k],
        s=12,
        linewidths=lw_pt,
        edgecolor='black'
    )

axs[0].set_xlabel(r'path / Å')
axs[0].set_ylabel('E / eV')

plot_band(neb_images, ax=axs[1], energy_reference=e_particle + isolated_ref['C'] + isolated_ref['O'])

axs[1].set_xlabel(r'path / Å')
axs[1].set_ylabel('E / eV')

for k in ['sum_two_body']:
    sns.scatterplot(
            data=res_df,
            x='path',
            y=k,
            label=label_dict[k],
            ax=axs[1],
            color=color_dict[k],
            s=12,
            linewidths=lw_pt,
            edgecolor='black',
            zorder=3,
        )
    

for k in ['d_body']:
    sns.scatterplot(
            data=res_df,
            x='path',
            y=k,
            label=label_dict[k],
            ax=axs[2],
            color=color_dict[k],
            s=12,
            linewidths=lw_pt,
            edgecolor='black',
            zorder=3,
        )

axs[2].axhline(
    y=res_df.d_body.values[-1],
    color="#9B8F8F",
    linewidth=.5,
    linestyle='--',
    zorder=1
)

axs[2].axvline(
    x=res_df.path.values[5],
    color="#b1008a",
    linewidth=.5,
    linestyle='-.',
    zorder=1
)

axs[2].axvline(
    x=res_df.path.values[3],
    color='#ff7f0e',
    linewidth=.5,
    linestyle='-.',
    zorder=1
)

axs[2].set_xlabel(r'path / Å')
axs[2].set_ylabel(r'ΔE / eV')
axs[2].set_xlim([-.1,6.5])
axs[1].set_xlim([-.1,6.5])
axs[0].set_xlim([-.1,6.5])

for ax in axs:
    ax.tick_params(labelsize=8)
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
axs[0].legend(fontsize=7, bbox_to_anchor=(1, 0.5))
axs[1].legend(fontsize=7, loc='upper right')
axs[2].legend(fontsize=7, loc='lower right')

fig.set_layout_engine(layout='tight')

### *O field in presence of *C

In [ ]:
#using a manifold instance but modifying the mesh data to be a plane in 
m_neb = Manifold(particle, mode='particle', precision=10, touch_sphere_size=2.5, wrap_on='sites')
verts, edges, faces, normals = generate_plane_mesh(images, resolution=0.05)

In [ ]:
# m_neb.calc = copy.deepcopy(clean_calc)
m_neb.grid = verts
m_neb.grid_atoms = Atoms(positions=verts, symbols=['X' for _ in verts])
m_neb.normals = normals
m_neb.faces = faces


In [ ]:

pop_keys = [k for k in m_neb.grid_atoms.arrays.keys() if 'grad' in k]

for k in pop_keys:
    m_neb.grid_atoms.arrays.pop(k)    

In [ ]:
verts, edges, faces, normals = generate_plane_mesh(images, resolution=0.02)
m_neb.grid = verts
m_neb.grid_atoms = Atoms(positions=verts, symbols=['X' for _ in verts])
m_neb.normals = np.array([(1,0,0) for _ in normals])
m_neb.faces = faces

In [ ]:
grds =[read(DATA_DIR / f'image_{i}_O_O_grid_prec0.02.xyz') for i in [3, 10]]#range(11)]
ref_grd  = read(DATA_DIR / f'image_0_OC_O_grid_prec0.02.xyz')

for i, grd in enumerate(grds):
    vals = grd.arrays['e_ClO'].flatten() - ref_grd.arrays['e_ClO'].flatten()
    vals-=vals[0]

    colors = [values_to_colors(v, [-2,2], reference_value=0, palette_nam="RdBu") for v in vals]

    filename = SCRATCH_DIR / f'image_{i}_dO_grid_prec0.02.ply'
    m_neb.save_ply(filename=filename, vertex_colors=colors)

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )

    # m_neb.save_ply(filename=f'{SCRATCH_DIR}/grd_O_image_{i}.ply', vertex_colors=colors)
    # m_neb.save_ply(filename=f'{SCRATCH_DIR}/grd_C_image_{i}.ply', vertex_colors=colors)

In [ ]:
symbol = 'C'

trace = Atoms()
for atoms in images:
    trace += atoms[atoms.symbols == symbol]
write(f'{SCRATCH_DIR}/neb_trace_{symbol}.xyz', trace)

In [ ]:
symbol = 'C'
lower_range,upper_range = -4, 4

ref_grd = read(f'../examples/CO_neb/particle_{symbol}_field_in_plane.xyz')


vals = ref_grd.arrays[f'e_Cl{symbol}'].flatten() 
# vals = ref_grd.arrays[f'e_ClC#O+'].flatten() 
vals-=vals[0]
vals[vals < lower_range] = lower_range
vals[vals > upper_range] = upper_range

colors = [values_to_colors(v, [lower_range,upper_range], reference_value=vals[0], palette_nam="plasma") for v in vals]

filename = f'{SCRATCH_DIR}/grd_{symbol}_ref.ply'
m_neb.save_ply(filename=filename, vertex_colors=colors)

subdivide_ply_smooth_color(
filename,
filename,
levels=1,
)

_a = ref_grd.copy()
_a.arrays[f'vals'] = vals 
write(f'{SCRATCH_DIR}/grd_{symbol}_ref.xyz', _a)


In [ ]:
subdivide_ply_smooth_color(
    SCRATCH_DIR / 'grd_delta_O_image_0.ply',
    SCRATCH_DIR / 'grd_delta_O_image_0_smooth.ply',
    levels=1,
    )

In [ ]:
_data = DATA_DIR / f'*_WRAPRatoms_PREC2.0_TSS3.0.xyz'
_data = _data.as_posix()
files = glob(_data) 
files.sort()

grid_dict = {}

for file in files:
    k = file.split('/')[-1].split('_')[1]
    grd = read(file)
    grid_dict[k] = grd


In [ ]:
# NOTE: ref_dict must be defined by running the 'ref_dict' cell below first.
mol = 'HN'

array_key = [_k for _k in grid_dict[mol].arrays.keys() if _k[:2] == 'e_'][0]
ens = grid_dict[f'{mol}'].arrays[array_key].copy()
pos = grid_dict[f'{mol}'].positions.copy()
ens -= ref_dict[f'e_{mol}']

### fragments morse

In [ ]:
smils = [
    'Cl[C]',
    'ClC',
    'S1SC([O-])C1',
    'S1SN=[NH+]1',
    'Cl[OH+]CC(O)CO',
    'Cl[n+]1ccccc1',
]

In [ ]:
site_container = read(DATA_DIR / 'grid_increments_vis.xyz')
atoms = particle.copy()

In [ ]:
############### make n vectors
sites_n_vectors = {
    0: site_container[-2].position - site_container[0].position,
    1: site_container[-1].position - site_container[1].position
}

sites_coords = {
    0: site_container[0].position,
    1: site_container[1].position
}

for k, v in sites_n_vectors.items():
    sites_n_vectors[k] = v/np.linalg.norm(v)

###############

tchsps = []
for touch_sphere_size in np.arange(1.5, 3.6, .5):
    for _ in [1,2]:
        tchsps+=[touch_sphere_size]  

# f = Fragment('Cl[C]', to_initialize=1).get_conformer(0)

data = []

for f in [Fragment(smi, to_initialize=1).get_conformer(0) for smi in smils]:
    
    for i, a in enumerate(site_container):
        site_i = (a.position[0]<11)*1

        _atoms = atoms.copy()
        attach_fragment(
            atoms = _atoms,
            site_dict = {'coordinates': a.position,'n_vector': sites_n_vectors[site_i],'h_vector': [1., 0., 0.]},
            fragment = f,
            n_rotation = 0,
            height = 0,
        ) 

        _atoms.calc = clean_calc
        info = {
            'site_i': site_i,
            'en': _atoms.get_potential_energy(),
            'manifold_dist': tchsps[i],
            'coordinates': a.position,
            'origin': 'points',
            'fragment': _atoms[len(atoms):].get_chemical_formula()
        }
        data.append(info)

    trj = []
    for d in np.arange(1.4, 3.6, 0.05):
        d -= 1.5
    # for d in np.arange(1.4, 3.6, 0.5):
        for i in [0,1]:
            _atoms = atoms.copy()
            attach_fragment(
                atoms = _atoms,
                site_dict = {'coordinates': sites_coords[i]+sites_n_vectors[i]*d,'n_vector': sites_n_vectors[i],'h_vector': [1., 0., 0.]},
                fragment = f,
                n_rotation = 0,
                height = 0,
            ) 
            _atoms.calc = copy.deepcopy(clean_calc)
            info = {
                'site_i': i,
                'en': _atoms.get_potential_energy(),
                'manifold_dist': tchsps[0] + d,
                'origin': 'line',
                'fragment': _atoms[len(atoms):].get_chemical_formula()
            }

            data.append(info)
            trj.append(_atoms)


data = pd.DataFrame(data)

refs = {}
for f in data.fragment.unique():
    refs[f] = data[data.fragment == f].en.min()

data['en_ref'] = [refs[f] for f in data.fragment.values]
data['en_rel'] = data['en'] - data['en_ref']

# data

In [ ]:
fig_w, fig_h = 85 / 25.4, 70 / 25.4
fig, axs = plt.subplots(2, 1, figsize=(fig_w, fig_h), dpi=600, sharex=True)

for i in data.site_i.unique():

    df = data[(data.origin == 'line') & (data.site_i == i) ]
    sns.lineplot(
        df, x='manifold_dist', y='en_rel', hue='fragment', ax=axs[i], legend=False,
        palette='plasma',
        zorder=3, linestyle='--'
        )

for i in data.site_i.unique():

    df = data[(data.origin == 'points') & (data.site_i == i) ]
    sns.scatterplot(
        df, x='manifold_dist', y='en_rel', hue='fragment', style='site_i', ax=axs[i], legend=False,
        palette='plasma',
        edgecolor='black',
        zorder=3
        )
    

# --- Formatting ---
for ax in axs:
    ax.set_xlim(1.4, 3.55)
    ax.set_ylim(-1, 6.5)
    ax.tick_params(width=0.4)
    ax.set_facecolor('white')

    # Thin axes spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

### manifold LSR
**Paper figure: Linear scaling relations on the manifold**

In [ ]:
atoms = particle.copy()
# view([p.get_conformer(0) for p in probes])

In [ ]:
_atoms = atoms.copy()
_atoms.calc= copy.deepcopy(clean_calc)
e_ref = _atoms.get_potential_energy()

ref_dict = {}

for p in probes:
    _atoms = p.get_conformer(0)[1:]
    _atoms.calc= copy.deepcopy(clean_calc)
    ref_dict['e_'+p.smile] = _atoms.get_potential_energy() + e_ref
    # ref_dict['e_'+p.get_chemical_formula()] = _atoms.get_potential_energy() + e_ref

In [ ]:
precision = 5
touch_sphere_size = 3.5


center = np.mean(atoms.positions, axis=0)
diffs = atoms.positions - center
dists = np.linalg.norm(diffs, axis=1)
index = np.argmax(dists)
particle_radius = dists[index]
grid_radius = particle_radius + touch_sphere_size + 0.5 # 0.5 is safety buffer
round_cube_geometry = grid_round_cube(center=center, radius=grid_radius, d_min=precision)


m_dict = {}

for touch_sphere_size in [1.5,2.,2.5,3.,3.5]:
# for touch_sphere_size in np.arange(2., 3.6, 0.1):
    
    m = Manifold(atoms, mode='particle', precision=precision, touch_sphere_size=touch_sphere_size, grid_mode = round_cube_geometry, wrap_on='sites')
    m.calc = copy.deepcopy(clean_calc)
    print(f'{touch_sphere_size = }, {len(m.grid)}')
    m.run_probe_scan(probes=probes)
    m_dict[touch_sphere_size] = m

In [ ]:
rows = []
o_and_c_and_n = {}

for touch_sphere_size, m in m_dict.items():

    data = {}
    for k in m.grid_atoms.arrays.keys():
        x = m.grid_atoms.arrays[k].flatten()
        if len(x) == len(m.grid_atoms) and 'grad' not in k:
            data[k] = x
        
        if k in ['e_Cl[C]', 'e_Cl[O]', 'e_Cl[N]']:
            o_and_c_and_n[k]=x
    
    df_atoms = pd.DataFrame(data)
    df_atoms['touch_sphere_size'] = touch_sphere_size
    rows.append(df_atoms)

df = pd.concat(rows, ignore_index=True)

for k in df.columns:
    if k in ref_dict.keys():
        df[k] = df[k] - ref_dict[k]


In [ ]:
# Identify all e_ columns
e_cols = [c for c in df.columns if c.startswith('e_')]

# Reference columns
ref_cols = ['e_Cl[C]', 'e_Cl[O]', 'e_Cl[N]']

# Non-reference e_ columns to melt
melt_cols = [c for c in e_cols if c not in ref_cols]

# Melt non-reference e_ columns
df_long = df.melt(
    id_vars=['numbers', 'touch_sphere_size'] + ref_cols,
    value_vars=melt_cols,
    var_name='probe',
    value_name='en'
)

# Keep both reference columns as separate en_ref columns
df_long = df_long.rename(columns={
    'e_Cl[C]': 'en_ref_C',
    'e_Cl[O]': 'en_ref_O',
    'e_Cl[N]': 'en_ref_N'
    })

df_long.head()

In [ ]:
formula_dict = {
    'e_ClC#[O+]': 'CO',
    'e_ClO':  'OH',
    'e_Cl[CH]':  'CH',
    'e_Cl[CH2]':  'CH2',
    'e_ClC':  'CH3',
    'e_Cl[N]':  'N',
    'e_Cl[NH]':  'NH',
    'e_ClN':  'NH2',
}

names_dict = {
    'e_ClC#[O+]': r'E$_{ads}$*CO / eV',
    'e_ClO': r'E$_{ads}$*OH / eV',
    'e_Cl[CH]': r'E$_{ads}$*CH / eV',
    'e_Cl[CH2]': r'E$_{ads}$*CH2 / eV',
    'e_ClC': r'E$_{ads}$*CH3 / eV',
    'e_Cl[N]': r'E$_{ads}$*N / eV',
    'e_Cl[NH]': r'E$_{ads}$*NH / eV',
    'e_ClN': r'E$_{ads}$*NH2 / eV'
}

fit_color = "#942A2A"

marker_colors = {
    'en_ref_N': "#0F0050",
    'en_ref_C':'#000000',
    'en_ref_O':"#570000",
}

df_long['formula'] = [
    formula_dict[x] for x in df_long.probe.values
]

In [ ]:
pdf = df_long[
    ~df_long.formula.str.contains('O')  &
    ~df_long.formula.str.contains('N')  &
    (df_long.touch_sphere_size > 1.5)
    ]

symbol = "C"
x_name = f'en_ref_{symbol}'



# unique facet values
probes = pdf['probe'].unique()
sizes = pdf['touch_sphere_size'].unique()

n_rows = len(probes)
n_cols = len(sizes)

inc = 182/25.4 / n_cols

fig, axs = plt.subplots(
    n_rows, n_cols,
    figsize= (inc* n_cols, inc *.85 * n_rows),
    # figsize= (3 * n_cols, 3 * n_rows),
    sharex='col',
    dpi=600
)

for i, probe in enumerate(probes):
    for j, size in enumerate(sizes):
        ax = axs[i, j]

        sub = pdf[
            (pdf['probe'] == probe) &
            (pdf['touch_sphere_size'] == size)
        ]

        sns.scatterplot(
            data=sub,
            x=x_name,
            y='en',
            ax=ax,
            color =marker_colors[x_name],
            alpha=.2,
            s=50
        )

        # extract data
        x = sub[x_name].values
        y = sub['en'].values

        # only fit if enough points
        if len(x) >= 2:
            # linear fit
            m, c = np.polyfit(x, y, 1)

            # draw fit line
            xfit = np.array([x.min(), x.max()])
            yfit = m * xfit + c

            ax.plot(
                xfit, yfit,
                color=fit_color,
                linewidth=1.,
                zorder=4
            )

            # text position: bottom-right
            xr = ax.get_xlim()
            yr = ax.get_ylim()


        
        # prediction + R^2
        y_pred = m * x + c
        r2 = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

        ax.text(
            0.98, 0.05,
            f'$R^2 = {r2:.3f}$\n$y = {m:.3f}x + {c:.3f}$',
            transform=ax.transAxes,
            ha='right',
            va='bottom',
            color=fit_color,
            fontsize=8
        )

        # titles / labels (FacetGrid-like)
        if i == 0:
            ax.set_title(f'Manifold distance = {size}')
            
        if j == 0:
            ax.set_ylabel(f'{names_dict[probe]}')
        else:
            ax.set_ylabel('')
        if i < n_rows - 1:
            ax.set_xlabel('')
        else:
            ax.set_xlabel(r'E$_{ads}$*'+f'{symbol} / eV')

fig.set_layout_engine(layout='tight')


In [ ]:
# fig_w, fig_h = 60 / 25.4, 150 / 25.4
# fig, axs = plt.subplots(3, 1, figsize=(fig_w, fig_h), dpi=60, sharey=False,sharex=False)

fig_w, fig_h = 182 / 25.4, 55 / 25.4
fig, axs = plt.subplots(1, 3, figsize=(fig_w, fig_h), dpi=600, sharey=False,sharex=False)



ax = axs[0]
pdf = df_long[
    ~df_long.formula.str.contains('C') &
    ~df_long.formula.str.contains('N') &
    (df_long.touch_sphere_size == 2.5)
    ]
sns.scatterplot(data=pdf, x='en_ref_O',
                y='en', hue='formula', style='formula', ax=ax, alpha=0.5, palette='plasma',
                edgecolor='black',linewidth=0.2, s = 20,legend=True
                )

add_linear_fits(
    ax=axs[0],
    df=pdf,
    xcol='en_ref_O',
    ycol='en',
    groupcol='formula'
)

ax = axs[1]
pdf = df_long[
    ~df_long.formula.str.contains('O') &
    ~df_long.formula.str.contains('C') &
    (df_long.touch_sphere_size == 2.5)
    ]
sns.scatterplot(data=pdf, x='en_ref_N',
                y='en', hue='formula', style='formula', ax=ax, alpha=0.5, palette='plasma',
                edgecolor='black',linewidth=0.2, s = 20,legend=True
                )

add_linear_fits(
    ax=axs[1],
    df=pdf,
    xcol='en_ref_N',
    ycol='en',
    groupcol='formula'
)

ax = axs[2]
pdf = df_long[
    ~df_long.formula.str.contains('O') &
    ~df_long.formula.str.contains('N') &
    (df_long.touch_sphere_size == 2.5)
    ]

sns.scatterplot(data=pdf, x='en_ref_C',
                y='en', hue='formula', style='formula', ax=ax, alpha=0.5, palette='plasma',
                edgecolor='black',linewidth=0.2, s = 20,legend=True
                )

add_linear_fits(
    ax=axs[2],
    df=pdf,
    xcol='en_ref_C',
    ycol='en',
    groupcol='formula'
)

tick_increment = 0.5  # e.g., 0.5 eV per tick

    
for ax in axs:

    ax.tick_params(width=0.4)
    ax.set_facecolor('white')
    ax.set_aspect('equal', adjustable='datalim')
    ax.xaxis.set_major_locator(MultipleLocator(tick_increment))
    ax.yaxis.set_major_locator(MultipleLocator(tick_increment))
    
    leg = ax.legend(loc='upper left')  # get the legend object
    
    # Thin axes spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


axs[0].set_xlabel(r'E$_{ads}$*O / eV')
axs[0].set_ylabel(r'E$_{ads}$*OH$_X$ / eV', fontsize=7)

axs[1].set_xlabel(r'E$_{ads}$*N / eV')
axs[1].set_ylabel(r'E$_{ads}$*NH$_X$ / eV', fontsize=7)

axs[2].set_xlabel(r'E$_{ads}$*C / eV')
axs[2].set_ylabel(r'E$_{ads}$*CH$_X$ / eV', fontsize=7)

fig.set_layout_engine(layout='tight')

# leg.get_frame().set_alpha(1)     # ← set solid background color

### LSR breaking map
**Paper figure: LSR-breaking map (exported as .ply for Blender rendering)**

In [ ]:
lsr_pairs = {
    'e_Cl[O]': ['e_ClO'],
    'e_Cl[N]': ['e_Cl[NH]', 'e_ClN'], 
    'e_Cl[C]': ['e_Cl[CH]', 'e_Cl[CH2]', 'e_ClC']
}

names_dict = {
    'e_ClC#[O+]': 'CO',
    'e_Cl': 'H',
    'e_Cl[O]': 'O',
    'e_ClO': 'OH',
    'e_Cl[C]': 'C',
    'e_Cl[CH]': 'CH',
    'e_Cl[CH2]': 'CH2',
    'e_ClC': 'CH3',
    'e_Cl[N]': 'N',
    'e_Cl[NH]': 'NH',
    'e_ClN': 'NH2'
    }



for system in ['HEO']:

    for x_key, y_keys in lsr_pairs.items():
        for y_key in y_keys:

            vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
            vals_x -= refs_dict[system][x_key]
            vals_y = grids_dict[system][y_key].arrays[y_key.replace('[','').replace(']','')+'_0'].copy()
            vals_y -= refs_dict[system][y_key]

            vals = fit_line_and_distances(vals_x, vals_y)
            color_range=[-.5,.5]

            print(f'{x_key = }, {y_key = }, {min(vals) = }, {max(vals) = }')
            
            colors = [
                values_to_colors(v, color_range, reference_value=None, palette_nam='RdBu') for v in vals
                ]
            
            sns.kdeplot(vals)
            filename=SCRATCH_DIR / f'LSR_break_map_{system}_{names_dict[x_key]}vs{names_dict[y_key]}.ply'

            # m.save_ply(
            #     filename=filename,
            #     vertex_colors=colors
            #     )
            
            # subdivide_ply_smooth_color(
            #     filename,
            #     filename,
            #     levels=1,
            # )



### Descriptor map
**Paper figure: Descriptor map Cu / Pd / Au / HEO**

In [ ]:
probes = [
    Fragment('ClC#[O+]', to_initialize=1),
    Fragment('Cl', to_initialize=1),
    Fragment('Cl[O]', to_initialize=1),
    Fragment('Cl[C]', to_initialize=1),
    Fragment('ClO', to_initialize=1),
    Fragment('Cl[CH]', to_initialize=1),
    Fragment('Cl[CH2]', to_initialize=1),
    Fragment('ClC', to_initialize=1),
    Fragment('Cl[N]', to_initialize=1),
    Fragment('Cl[NH]', to_initialize=1),
    Fragment('ClN', to_initialize=1),
    ]


metals = {
    "Cu": 3.615,
    "Pd": 3.889,
    "Au": 4.078
}

slabs = {}

for metal, a in metals.items():
    slab = fcc211(
        symbol=metal,
        size=(6, 3, 4),   # (x, y, layers)
        a=a,
        vacuum=15.0,
    )

    slab.set_pbc([True, True, False])
    slabs[metal] = slab

for metal, slab in slabs.items():
    write(SCRATCH_DIR / f'metal_{metal}_211_slab.xyz', slab)

In [ ]:
precision = 1
touch_sphere_size = 2.5

# Calculate manifold probe scan on pristine metals on the fly

for metal in slabs.keys():
    m = Manifold(slabs[metal],
                mode='slab',
                precision=precision,
                touch_sphere_size=touch_sphere_size,
                grid_mode = round_cube_geometry, wrap_on='atoms')
    m.calc = copy.deepcopy(clean_calc)
    m.run_probe_scan(probes=probes)
    m.write_grid(DATA_DIR / f'metal_{metal}_211_grid.xyz')


In [ ]:
# Note: 'shuang_' prefix is a data-provenance naming artifact from the original dataset files.
sstring = DATA_DIR / 'shuang*.xyz'
sstring = sstring.as_posix()
for file in glob(sstring):
    print(f"'{[k for k in read(file).arrays.keys() if k[:2] == 'e_'][0]}': read('{file}'),")

In [ ]:
refs_dict = {}

for metal in slabs_dict.keys():
    refs_dict[metal] = {}
    for probe in probe_dict.keys():
        refs_dict[metal][probe] = slab_refs[metal] + probe_dict[probe]

refs_dict

In [ ]:
fig, axs = plt.subplots(2,1, figsize = [77/25.4, 77/26.4], dpi=600, sharex=True)

color_dict = {
    'HEA':"#686868",
    'HEO': "#000000",
    'Cu': "#E97100",
    'Au': "#FFF897",
    'Pd': "#C0D1FF",
}

name_dict = {
    'e_ClC#[O+]':"CO",
    'e_Cl':"H",
    'e_Cl[O]':"O",
}


for i, y_key in enumerate(['e_ClC#[O+]', 'e_Cl[O]']):
    ax = axs[i]
    x_key = 'e_Cl'
    # y_key = 'e_ClC#[O+]'


    for system in [ 'Au', 'Pd', 'Cu']:

        vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
        vals_x -= refs_dict[system][x_key]
        vals_y = grids_dict[system][y_key].arrays[y_key.replace('[','').replace(']','')+'_0'].copy()
        vals_y -= refs_dict[system][y_key]

        sns.scatterplot(
            x= vals_x,
            y =vals_y,
            color=color_dict[system],
            label=f"{system} (211)",
            alpha=.8,
            ax=ax,
            # levels=3
            # fill=True
            s=40
        )

    ax = axs[i]
    for system in ['HEO']: #, 'HEA'

        vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
        vals_x -= refs_dict[system][x_key]
        vals_y = grids_dict[system][y_key].arrays[y_key.replace('[','').replace(']','')+'_0'].copy()
        vals_y -= refs_dict[system][y_key]

        sns.kdeplot(
            x= vals_x,
            y =vals_y,
            color=color_dict[system],
            label=system,
            alpha=.9,
            ax=ax,
            linestyles='--', linewidths=0.5,
            levels=6
        )


    # for ax in axs:
    axs[i].set_xlabel(r'E$_{ads}$*'+f'{name_dict[x_key]} / eV', fontsize=7)
    axs[i].set_ylabel(r'E$_{ads}$*'+f'{name_dict[y_key]} / eV', fontsize=7)
    leg = axs[i].legend(loc='upper left')  # get the legend object

axs[0].get_legend().remove()
axs[0].set_xlim(-1.6, 0.2)
axs[0].set_ylim(-1.1, 1.1)

axs[1].set_xlim(-1.6, 0.2)
axs[1].set_ylim(-2.5, .5)



tick_increment = 0.5  # e.g., 0.5 eV per tick
    
# for ax in axs:

# #     # ax.set_xlim(-2.1, -.1)
# #     # ax.set_ylim(-2.1, -.1)
# #     ax.tick_params(width=0.4)
# #     ax.set_facecolor('white')
#     ax.set_aspect('equal', adjustable='datalim')
#     ax.xaxis.set_major_locator(MultipleLocator(tick_increment))
#     ax.yaxis.set_major_locator(MultipleLocator(tick_increment))
    
#     # ax.set_ylabel(r'E$_{ads}$*AH$_X$ / eV', fontsize=7)
    
    
#     # Thin axes spines
#     for spine in ax.spines.values():
#         spine.set_linewidth(0.8)



fig.set_layout_engine(layout='tight')

In [ ]:
atoms = read(DATA_DIR / 'Co0.2Ni0.2Mg0.2Zn0.2Mn0.2Al2O4COH_311_cond2.traj')
atoms = atoms[[atom.index for atom in atoms if atom.symbol != 'X']]
# view(atoms)

In [ ]:
wrap_on='atoms'#'sites'
# precision=0.3
precision=.2
touch_sphere_size=2.5

m_heo = Manifold(
        atoms.copy(),
        mode='slab',
        precision=precision,
        touch_sphere_size=touch_sphere_size,
        wrap_on=wrap_on,
        calc=copy.deepcopy(clean_calc)
        )

In [ ]:
import math

In [ ]:
refs_dict

In [ ]:
# Note: 'shuang_' prefix is a data-provenance naming artifact from the original dataset files.
name_dict = {
    'e_ClC#[O+]':"CO",
    'e_Cl':"H",
    'e_Cl[O]':"O",
}


    # x_key = 'e_Cl'
for x_key in list(name_dict.keys()):
    
    system =  'HEO'
    filename = SCRATCH_DIR / f'shuang_{system}_{x_key}.ply'
    filename = SCRATCH_DIR / f'shuang_{system}_{x_key}_full_range.ply'

    
    vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
    vals_x -= refs_dict[system][x_key]
    # sns.histplot(vals_x)
    color_range = [math.floor(np.min(vals_x)*10)/10, math.ceil(np.max(vals_x)*10)/10]
    print(f'{x_key = }, {color_range = }')
    
    color_range = [-3.6, 1.0]
    colors = [values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in vals_x]


    m_heo.save_ply(filename=filename, vertex_colors=colors)

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )

    # m_heo.save_ply(filename=SCRATCH_DIR / 'x.ply', vertex_colors=colors)

In [ ]:
# Note: 'shuang_' prefix is a data-provenance naming artifact from the original dataset files.
name_dict = {
    'e_Cl[O]':"O",
    'e_Cl':"H",
    'e_ClC#[O+]':"CO",
}

system =  'HEO'
# filename = SCRATCH_DIR / f'shuang_{system}_RGB_full_range.ply'
filename = SCRATCH_DIR / f'shuang_{system}_RGB_scaled.ply'


rgb_vals = []

for x_key in list(name_dict.keys()):

    vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
    vals_x -= refs_dict[system][x_key]
    sns.histplot(vals_x)
    
    color_range = [math.floor(np.min(vals_x)*10)/10, math.ceil(np.max(vals_x)*10)/10]
    # color_range = [-3.6, 1.0]

    span = color_range[1] - color_range[0]
    vals_x -= np.min(vals_x)
    print(f'{x_key = }; {span = }')
    vals_x = [int((1-(v/span)) * 255) for v in vals_x]

    rgb_vals.append(vals_x)

    
    
    colors = np.array(rgb_vals).T

    # colors = [values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in vals_x]


m_heo.save_ply(filename=filename, vertex_colors=colors)

subdivide_ply_smooth_color(
    filename,
    filename,
    levels=1,
)

### Multi-channel descriptor color code

In [ ]:
N = 255  # subdivisions per edge

verts = []
cols = []
faces = []
vid = {}

def add_vertex(p):
    key = tuple(p)
    if key not in vid:
        vid[key] = len(verts)
        verts.append(p)
        cols.append((255 * p).astype(int))
    return vid[key]

def quad(a, b, c, d):
    faces.append([a, b, c, d])

lin = np.linspace(0, 1, N + 1)

# generate cube faces
for axis in range(3):
    for side in [0, 1]:
        for i in range(N):
            for j in range(N):
                p = np.zeros(3)
                q = np.zeros(3)
                r = np.zeros(3)
                s = np.zeros(3)

                p[axis] = side
                q[axis] = side
                r[axis] = side
                s[axis] = side

                a, b = (axis + 1) % 3, (axis + 2) % 3

                p[a], p[b] = lin[i],   lin[j]
                q[a], q[b] = lin[i+1], lin[j]
                r[a], r[b] = lin[i+1], lin[j+1]
                s[a], s[b] = lin[i],   lin[j+1]

                quad(
                    add_vertex(p),
                    add_vertex(q),
                    add_vertex(r),
                    add_vertex(s)
                )

# write PLY
with open(str(DATA_DIR) + "/cube_rgb.ply", "w") as f:
    f.write("ply\nformat ascii 1.0\n")
    f.write(f"element vertex {len(verts)}\n")
    f.write("property float x\nproperty float y\nproperty float z\n")
    f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
    f.write(f"element face {len(faces)}\n")
    f.write("property list uchar int vertex_indices\nend_header\n")

    for v, c in zip(verts, cols):
        f.write(f"{v[0]} {v[1]} {v[2]} {c[0]} {c[1]} {c[2]}\n")

    for fa in faces:
        f.write(f"4 {fa[0]} {fa[1]} {fa[2]} {fa[3]}\n")


### PMD distribution setup

In [ ]:
scale = 100
poss = [x/scale for x in range(0,5*scale)]

ens = []
for pos in poss:
    x = Atoms(['C','O'], [[0,0,0],[pos,0,0]])
    x.calc = copy.deepcopy(clean_calc)
    ens.append(x.get_potential_energy())


In [ ]:
m = Manifold(
    particle.copy(),
    mode='particle',
    precision=5,
    touch_sphere_size=2.,
    wrap_on='blend',
    calc=copy.deepcopy(clean_calc)
    )

In [ ]:
m.run_probe_scan(probes=[get_neb_probe(PMD_image, keep=['C'])], n_rotation=180)
m.grid_atoms.arrays['e_S1S[C][O]1']

In [ ]:
PMD_image = PMD_image[[atom.index for atom in PMD_image if atom.symbol in ['C','O']]]
PMD_image.get_distance(0,1)

### Visualize PMD grid structures

In [ ]:
angles = [phi for phi in range(0,360, 10)]
trj = [Atoms() for _ in angles]

for i, point in enumerate(m.grid):
# for i, point in [(16643, m.grid[16643])]:
    for i_phi, phi in enumerate(angles):
    # for i_phi, f in enumerate(fs):

        x = Atoms()        

        attach_fragment(
            x,
            site_dict = {
                    'coordinates': point,
                    'n_vector': m.normals[i],
                },
            fragment = get_neb_probe(PMD_image, keep=['C','O']).get_conformer(0),
            n_rotation = phi,
            height = 0,
        )
        trj[i_phi]+=Atoms(['He' for _ in range(100)], [point+_d/40*m.normals[i] for _d in range(100)])
        trj[i_phi]+=x
        # trj.append(x)
# trj += particle.copy()

write(SCRATCH_DIR / 'anisotropic_toy.xyz', trj)

In [ ]:
site_atoms = trj[0][[atom.index for atom in trj[0] if atom.symbol == 'X']]
for atom in site_atoms:
    atom.symbol = 'He'
write(SCRATCH_DIR / 'site.xyz',  site_atoms)

heli_trj = [a[[atom.index for atom in a if atom.symbol in ['C','O']]] for a in trj]
write(SCRATCH_DIR / 'site_helicopter.xyz',  heli_trj)

### Anisotric PMD calculations
**Paper figure: Anisotropy estimate (PMD φ-sweep)**

In [ ]:
m = Manifold(
    particle.copy(),
    mode='particle',
    precision=.2,
    touch_sphere_size=2.,
    wrap_on='blend',
)

In [ ]:
# key = 'TS' 
keys = ['CO', 'C', 'O']

aniso_dict = dict((key, []) for key in keys)
aniso_trj = []

for phi in range(0,360,10):
    # try:
    grid = phi_grid_dict['CO'][phi].copy()
    for key in keys:
        
        _atoms = phi_grid_dict[key][phi]
        ens = _atoms.arrays['e_S1SCO1_0']  - isolated_ref[key] - e_particle
        if key =='CO':
            ens -= e_particle
        aniso_dict[key].append(ens)
        grid.arrays[f'aniso_{key}'] = ens
    aniso_trj.append(grid)
    # except:
    #     break


for key in keys:
    aniso_std = np.std(aniso_dict[f'{key}'], axis=0) 
    aniso_e_min = np.min(aniso_dict[f'{key}'], axis=0) 

    aniso_dict[f'{key}_std'] = aniso_std
    aniso_dict[f'{key}_e_min'] = aniso_e_min
    # aniso_e_min = np.min(aniso,axis=0)

    for i, a in enumerate(aniso_trj):
        a.arrays[f'{key}_aniso_std'] = aniso_std
        a.arrays[f'{key}_aniso_e_min'] = aniso_e_min

PMD_FIELD = np.array(aniso_dict['CO']) - np.array(aniso_dict['C']) - np.array(aniso_dict['O']) - isolated_ref['PMD_CO']#PMD_image.get_potential_energy()
for i, a in enumerate(aniso_trj):
    a.arrays[f'PMD-CF_aniso'] = PMD_FIELD[i,:]
    a.arrays[f'PMD-CF_aniso_emin'] = np.min(PMD_FIELD, axis=0)

In [ ]:
aniso_trj =read(DATA_DIR / 'anisotropic_reactivity_map_CO.xyz', index=':')

In [ ]:
ranges = {}
ranges['O_aniso_e_min'] = [-4, -1]; ranges['aniso_O'] = [-4, -1]
ranges['C_aniso_e_min'] = [-6, -1]; ranges['aniso_C'] = [-6, -1]
ranges['CO_aniso_e_min'] = [-12, -10]; ranges['aniso_CO'] = [-12, -10]
ranges['PMD-CF_aniso_emin'] = [8, 11]; ranges['PMD-CF_aniso'] = [8, 11] 

ranges['O_aniso_std'] = [0., 1.2]
ranges['C_aniso_std'] = [0., 1.2]
ranges['CO_aniso_std'] = [0., 1.2]

ranges


In [ ]:
system = 'HEA'
for key, color_range in ranges.items():

    vals = aniso_trj[5].copy().arrays[key]
    colors = [
    values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in vals
    ]

    filename=SCRATCH_DIR / f'ANISO_isolated_ref_{key}--{system}.ply'

    m.save_ply(
        filename=filename,
        vertex_colors=colors
        )

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )

In [ ]:
vas_o = read(DATA_DIR / f'manifold_O_WRAPONblend_PREC0.2_TSS2.0.xyz').arrays['e_ClO_0']
vas_o -= e_particle
vas_o -= isolated_ref['O']
vas_c = read(DATA_DIR / f'manifold_C_WRAPONblend_PREC0.2_TSS2.0.xyz').arrays['e_ClC_0'] 
vas_c -= e_particle
vas_c -= isolated_ref['C']
vas_co = read(DATA_DIR / f'manifold_CO_WRAPONblend_PREC0.2_TSS2.0.xyz').arrays['e_ClC#O+_0'] 
vas_co -= e_particle
vas_co -= isolated_ref['C']
vas_co -= isolated_ref['O']

vals_dict = {
    # 'O': vas_o,
    # 'C': vas_c,
    'CO': vas_co,
}

ranges = {
    'O': [-3.5, -1],
    'C': [-4.5, -2],
    'CO': [-12.5, -10.5],    
}


system = 'HEA'
for key, color_range in ranges.items():

    vals = vals_dict[key]
    colors = [
    values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in vals
    ]

    filename=SCRATCH_DIR / f'MANIFOLD_isolated_ref_{key}--{system}.ply'

    m.save_ply(
        filename=filename,
        vertex_colors=colors
        )

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )

In [ ]:
system = 'HEA'
for key, color_range in ranges.items():

    vals = aniso_trj[5].copy().arrays[key]
    colors = [
        values_to_colors(
            v,
            color_range,
            reference_value=None,
            palette_nam='plasma'
        ) for v in vals
    ]

    filename=SCRATCH_DIR / f'MANIFOLD_isolated_ref_{key}--{system}.ply'

    m.save_ply(
        filename=filename,
        vertex_colors=colors
        )

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )

In [ ]:
delta = 0.1
distance = .9

angles = [theta for theta in range(0,360, 30)]

c_key = 'e_Cl[C]'
o_key = 'e_Cl[O]'
system = 'HEA'
vals_c = grids_dict[system][c_key].arrays[c_key.replace('[','').replace(']','')+'_0'].copy()
vals_c -= refs_dict[system][c_key]
vals_o = grids_dict[system][o_key].arrays[o_key.replace('[','').replace(']','')+'_0'].copy()
vals_o -= refs_dict[system][o_key]
    


for theta in angles:
    anisotropy = []
    for i, p0 in enumerate(m.grid):
        d_grid = np.linalg.norm(m.grid - p0, axis=1)

        anisotropy_mask = (d_grid < distance + delta) & (d_grid > distance - delta)

        point_inds =  np.where(anisotropy_mask)[0]
        points = m.grid[point_inds]
        normal = m.normals[i]

        furthest_pairs = points_close_to_rotating_line(
                p0,
                normal,
                points,
                angles = [theta],
                tol=.7
            )
        de = []
        for pair in furthest_pairs:
            v = np.min(vals_c[pair[0]]) + np.min(vals_o[pair[1]])
            anisotropy.append(v)
        
            
    color_range=[-4,-3.2]

    colors = [
        values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in anisotropy
        ]

    filename=SCRATCH_DIR / f'anisotropy_delta{delta}_distance{distance}_{theta}_{system}_CvsO.ply'

    m.save_ply(
        filename=filename,
        vertex_colors=colors
        )

    subdivide_ply_smooth_color(
        filename,
        filename,
        levels=1,
    )
            
        

In [ ]:
system = 'HEA'
color_range=[-4,-3.2]

colors = [
    values_to_colors(v, color_range, reference_value=None, palette_nam='plasma') for v in anisotropy
    ]

filename=SCRATCH_DIR / f'anisotropy_std_0.9_delta1.7_{system}_CvsO.ply'

m.save_ply(
    filename=filename,
    vertex_colors=colors
    )

subdivide_ply_smooth_color(
    filename,
    filename,
    levels=1,
)

In [ ]:
fig, axs = plt.subplots(1,2, figsize = [125/25.4, 90/26.4], dpi=600, sharey='row')

for system in ['HEO', 'HEA']:

    vals_x = grids_dict[system][x_key].arrays[x_key.replace('[','').replace(']','')+'_0'].copy()
    vals_x -= refs_dict[system][x_key]
    vals_y = grids_dict[system][y_key].arrays[y_key.replace('[','').replace(']','')+'_0'].copy()
    vals_y -= refs_dict[system][y_key]

    sns.kdeplot(
        x= vals_x,
        y =vals_y,
        color=color_dict[system],
        label=system,
        alpha=.9,
        ax=ax,
        linestyles='--', linewidths=0.5,
        levels=5
    )


# for ax in axs:
axs[0].set_xlabel(r'E$_{ads}$*'+f'{name_dict[x_key]} / eV', fontsize=7)
axs[0].set_ylabel(r'E$_{ads}$*'+f'{name_dict[y_key]} / eV', fontsize=7)
# axs[0].set_xlim(-1.6, -0.1)
# axs[0].set_ylim(-1.5, 1)



tick_increment = 0.5  # e.g., 0.5 eV per tick
    
for ax in axs:

#     # ax.set_xlim(-2.1, -.1)
#     # ax.set_ylim(-2.1, -.1)
#     ax.tick_params(width=0.4)
#     ax.set_facecolor('white')
    ax.set_aspect('equal', adjustable='datalim')
    ax.xaxis.set_major_locator(MultipleLocator(tick_increment))
    ax.yaxis.set_major_locator(MultipleLocator(tick_increment))
    
#     # ax.set_ylabel(r'E$_{ads}$*AH$_X$ / eV', fontsize=7)
#     leg = ax.legend(loc='upper left')  # get the legend object
    
#     # Thin axes spines
#     for spine in ax.spines.values():
#         spine.set_linewidth(0.8)



fig.set_layout_engine(layout='tight')

### BEP benchmark

In [ ]:
#read csv generated by neb_test.py
df_reaction_info = pd.read_csv(f'{DATA_DIR}/df_reaction_info.csv')

In [ ]:
len(df_reaction_info)

In [ ]:
# NOTE: construct_trajectory is a local helper not part of the public cft package
from cft.neb_utils import construct_trajectory

pmd_image_full = neb_images[5].copy()
x= get_neb_probe(pmd_image_full, keep=['C', 'O'])
x.smile = 'ClC#[O+]'
f_pmd = x.get_conformer(0)

best_pathway_row = df_reaction_info.iloc[0] # dataframe is already sorted by E_barrier!
traj = construct_trajectory(best_pathway_row, m, particle, f_pmd)
# view(traj)

In [ ]:
idx = df_reaction_info.groupby(['site','angle'])['vals_PMD'].idxmin()
xdf = df_reaction_info.loc[idx].reset_index(drop=True)

In [ ]:
len(xdf)

In [ ]:
fig, axs = plt.subplots(3,2, figsize = [92/25.6, 115/25.6], dpi=600)#, sharey='row', sharex='row')

# pdf = df_reaction_info

# pdf = xdf 
# pdf = df_reaction_info#[df_reaction_info.first_neighbor == 'Au']

# sns.scatterplot(data=pdf, x='E_reaction', y='E_barrier', hue='first_neighbor', ax=axs, alpha=0.1, legend=True)

axs = axs.flatten()

# make first 4 share x and y
for i in range(1, 4):
    axs[i].sharex(axs[0])
    axs[i].sharey(axs[0])

for i, metal in enumerate(['Au', 'Cu', 'Pt', 'Ag']):

    ax = axs[i]

    pdf = df_reaction_info[df_reaction_info.first_neighbor.isin(['Pd'])]
    
    sns.scatterplot(
        data=pdf,
        x='E_reaction',
        y='E_barrier',
        hue='first_neighbor',
        palette=metal_colors,
        s=0.5,
        ax=ax,
        alpha=1,
        legend=False,
        linewidth=0,
    )

    # pdf = xdf[xdf.first_neighbor.isin([metal])]
    pdf = df_reaction_info[df_reaction_info.first_neighbor.isin([metal])]
    
    sns.scatterplot(
        data=pdf,
        x='E_reaction',
        y='E_barrier',
        hue='first_neighbor',
        palette=metal_colors,
        s=2.,
        ax=ax,
        alpha=0.05,
        legend=False,
        linewidth=0
    )

    ax.set_xlabel(r'$\Delta E_{\text{react.}}^{\text{PMD}}$ / eV')
    ax.set_ylabel(r'$\Delta E_{\text{HBO}}^{\mathcal{M}}~-~E_{\text{react.}}^{\text{IS}}$ / eV')


ax = axs[-2]

pdf = df_reaction_info

sns.scatterplot(
    data=pdf,
    x='vals_FS',
    y='vals_IS',
    hue='first_neighbor',
    palette=metal_colors,
    s=0.5,
    ax=ax,
    alpha=0.2,
    legend=False,
    linewidth=0
)
ax.set_xlabel(r'$E_{\text{react.}}^{\text{FS}}$ / eV')
ax.set_ylabel(r'$E_{\text{react.}}^{\text{IS}}$ / eV')


ax = axs[-1]

pdf = df_reaction_info

sns.scatterplot(
    data=pdf,
    x='vals_FS',
    y='vals_PMD',
    hue='first_neighbor',
    palette=metal_colors,
    s=0.5,
    ax=ax,
    alpha=0.2,
    legend=False,
    linewidth=0
)

ax.set_xlabel(r'$E_{\text{react.}}^{\text{FS}}$ / eV')
ax.set_ylabel(r'$\Delta E_{\text{HBO}}^{\mathcal{M}}$ / eV')

# axs[-2].set_xlabel(r'$\Phi_{\text{PMD},~\alpha~=~\text{const}}^{\mathcal{M}}$')
# axs[-2].set_ylabel(r'$min_{\alpha}(\Phi_{\text{PMD}}^{\mathcal{M}})$')
# axs[-1].set_xlabel(r'$\sigma_{\alpha}(\Phi_{\text{PMD}}^{\mathcal{M}})$')
# axs[-1].set_ylabel(r'$E_{\text{react.}}^{\text{FS}}$ / eV')


# plt.legend(title="Site")
fig.set_layout_engine(layout='tight')


In [ ]:
from cft.plot_utils import plot_metal_bep_panels


In [ ]:
pdf = df_reaction_info#[df_reaction_info.first_neighbor == 'Au']

fig, axs = plot_metal_bep_panels(
    pdf, xcol='vals_FS',
    ycol='vals_PMD',
    metal_col='first_neighbor', 
    palette=metal_colors,
    alpha=0.15, s=20,
    add_fits=True
    )


In [ ]:
from cft.plot_utils import plot_metal_distributions
import matplotlib.pyplot as plt

fig, ax = plot_metal_distributions(
    df_reaction_info, 
    val_col='vals_PMD', 
    metal_col='first_neighbor', 
    palette=metal_colors,
    s=5  # tweak maker size depending on dataset density
)

# plt.savefig('SI_metal_distributions.pdf', dpi=600, bbox_inches='tight')
plt.show()


### prep NEB structures

In [ ]:
f = Fragment('ClC#[O+]', to_initialize=1).get_conformer(0)

neb_seed_lst = []
for i in xdf.index.values:

    row = xdf.iloc[i].to_dict()
    
    IS_atoms = particle.copy()
    i_c = int(row['i_c'])
    i_o = int(row['i_o'])
    IS_atoms += Atoms(['C'], [m.grid[i_c]])
    IS_atoms += Atoms(['O'], [m.grid[i_o]])

    i_pdm = int(row['i_pmd'])
    PMD_atoms = particle.copy()
    _pdm = get_PMD_structure(m.grid[i_pdm], m.normals[i_pdm], row['angle'], fragment=f_PMD_image.get_conformer(0))
    PMD_atoms += _pdm
    
    FS_atoms = particle.copy()
    i_co = int(row['i_pmd'])
    attach_fragment(
        FS_atoms,
        site_dict = {
                'coordinates': m.grid[i_co],
                'n_vector': m.normals[i_co],
            },
        fragment = f,
        n_rotation = phi,
        height = 0
    )

    neb_seed_lst += [IS_atoms, PMD_atoms, FS_atoms]

write(fstr(SCRATCH_DIR / 'neb_seed_lst.xyz'), neb_seed_lst)

## plot spectral
**Paper figure: Spectral KDE heatmap**

In [ ]:
relaxed_trj = read(str(DATA_DIR / 'HEA_toy') + '/gly_spectral/out_traj.xyz', index=':')

data = pd.DataFrame([a.info for a in relaxed_trj])
e_ref = data[data.d > (data.d.max() - 0.01)].en.max()
data['en_rel'] = data['en'] - e_ref


In [ ]:

# --- Adjustable Parameters ---
figsize = (71/25.4, 87.629/25.4)  # width, height in inches
fontsize = 7       # font size for labels and ticks
rows_per_bar = 10    # height of each KDE stripe
gap = 5              # vertical space between stripes
E_bins = 700         # Number of points in the energy grid

# --- Explicitly Set Constant Kernel Standard Deviation (Sigma) ---
constant_kernel_sigma = 0.02 # This is the "sigma" you want for each kernel

# --- Data Preparation ---
d_unique = np.sort(data['d'].unique())

# --- Energy range ---
E_min = -1.2
E_max = 0.2
E_grid = np.linspace(E_min, E_max, E_bins)

# --- Manual Gaussian KDE Function ---
def manual_gaussian_kde(data_points, evaluation_grid, kernel_sigma):
    """
    Performs a 1D Gaussian Kernel Density Estimation manually.
    """
    if len(data_points) == 0:
        return np.zeros_like(evaluation_grid)

    density = np.zeros_like(evaluation_grid, dtype=float)
    
    # Calculate pre-factors for the Gaussian function
    # Gaussian PDF: (1 / (sigma * sqrt(2 * pi))) * exp(-0.5 * ((x - mu) / sigma)^2)
    # We sum densities, so the (1/N) normalization usually applied after summation.
    # We normalize each row to max 1 later, so the constant pre-factor (1 / (sigma * sqrt(2 * pi)))
    # can be applied once after summation or effectively skipped if we normalize to max 1 anyway.
    # For now, let's keep it simple and sum un-normalized Gaussians, then normalize the total.

    for x_i in data_points:
        # Add a Gaussian centered at x_i to the density
        # The (x - mu)^2 part
        exponent = -0.5 * ((evaluation_grid - x_i) / kernel_sigma)**2
        # np.exp(exponent) is the un-normalized Gaussian shape
        density += np.exp(exponent)
    
    # Optional: Initial normalization by number of points to get actual density values
    # if len(data_points) > 0:
    #    density /= len(data_points)
    # The (1 / (sigma * sqrt(2 * pi))) factor could also be applied here
    # density /= (kernel_sigma * np.sqrt(2 * np.pi)) * len(data_points)

    return density

# --- Heatmap initialization ---
heatmap_height = len(d_unique) * (rows_per_bar + gap) - gap
heatmap = np.ones((heatmap_height, E_bins)) * np.nan

# --- Build KDE for each distance and populate heatmap ---
for i, di in enumerate(d_unique):
    Ei_values = data.loc[data['d'] == di, 'en_rel'].to_numpy()

    # Perform manual KDE
    density = manual_gaussian_kde(Ei_values, E_grid, constant_kernel_sigma)
    
    # Handle cases where density might be all zeros (e.g., no data points, or points outside grid)
    if np.max(density) > 0:
        density /= np.max(density)  # Normalize density to [0, 1] per row
    else:
        # If max density is 0, it means no meaningful KDE, so set to zeros
        density = np.zeros_like(E_grid)

    row_start = i * (rows_per_bar + gap)
    row_end = row_start + rows_per_bar
    if row_end > heatmap_height:
        row_end = heatmap_height
    heatmap[row_start:row_end, :] = np.tile(density, (row_end - row_start, 1))

# --- Plotting ---
fig, ax = plt.subplots(figsize=figsize, dpi=600)

sns.heatmap(
    heatmap,
    ax=ax,
    cmap='plasma',
    cbar=True,
    linewidths=0,
    linecolor='none',
    mask=np.isnan(heatmap)
)

# --- Add Black Box Outlines ---
for i in range(len(d_unique)):
    y0 = i * (rows_per_bar + gap)
    rect = Rectangle(
        (0, y0),
        E_bins,
        rows_per_bar,
        linewidth=0.8,
        edgecolor='black',
        facecolor='none'
    )
    ax.add_patch(rect)

# --- Flip Y axis (lowest d at bottom) ---
ax.invert_yaxis()

# --- Set Energy (X) Ticks ---
tick_step_x = 0.5
E_ticks = np.arange(E_min, E_max + tick_step_x/2, tick_step_x)
x_tick_idx = [np.argmin(np.abs(E_grid - t)) for t in E_ticks]
ax.set_xticks(x_tick_idx)
ax.set_xticklabels([f"{x:.1f}" for x in E_ticks], fontsize=fontsize)
ax.set_xlabel("Energy (en_rel)", fontsize=fontsize)

# --- Set Distance (Y) Ticks ---
y_tick_idx = [i * (rows_per_bar + gap) + rows_per_bar / 2 for i in range(len(d_unique))]
ax.set_yticks(y_tick_idx)
ax.set_yticklabels([f"{val:.2f}" for val in d_unique], fontsize=fontsize)
ax.set_ylabel("Distance (d)", fontsize=fontsize)

# --- Title ---
ax.set_title(f"KDE Heatmap of Energy per Distance (Manual Kernel Sigma: {constant_kernel_sigma})", fontsize=fontsize, pad=10)

# --- Adjust Colorbar ---
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=fontsize)
cbar.set_label('Normalized Density', fontsize=fontsize)

plt.tight_layout()
plt.show()

In [ ]:
data1 = pd.read_csv(str(DATA_DIR / 'HEA_toy') + '/N2H_anisotropic/data_4147.csv')
data2 = pd.read_csv(str(DATA_DIR / 'HEA_toy') + '/N2H_anisotropic/data_3469.csv')

for data in [data1,data2]:
    data['en_rel'] = data['en']-data.en.values[-1]

In [ ]:
atoms1 = read(str(DATA_DIR / 'HEA_toy') + '/N2H_anisotropic/out_traj_4147.xyz', index=1500)
atoms2 = read(str(DATA_DIR / 'HEA_toy') + '/N2H_anisotropic/out_traj_3469.xyz', index=1500)

In [ ]:
write(SCRATCH_DIR / 'top_1500.xyz', atoms2)
write(SCRATCH_DIR / 'bridge_1500.xyz', atoms1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
figsize_mm = (36, 16)  # width, height in millimeters
figsize_inch = (figsize_mm[0] / 25.4, figsize_mm[1] / 25.4)
vmin, vmax = -1, 0
cmap = "plasma"
angle_in_degrees = True  # False if already radians
highlight_index = 1500   # index of data point to mark

def make_polar(ax, df):
    r_vals = df["r"].to_numpy()
    d_vals = df["d"].to_numpy()
    en_vals = df["en_rel"].to_numpy()

    if angle_in_degrees:
        r_vals = np.deg2rad(r_vals)

    # --- Grid ---
    r_bins, d_bins = 300, 300
    r_grid = np.linspace(r_vals.min(), r_vals.max(), r_bins)
    d_grid = np.linspace(d_vals.min(), d_vals.max(), d_bins)
    R, D = np.meshgrid(r_grid, d_grid)

    # --- Interpolate energy onto grid ---
    Z = griddata((r_vals, d_vals), en_vals, (R, D), method="linear", fill_value=np.nan)

    # --- Plot heatmap ---
    ax.pcolormesh(R, D, Z, cmap=cmap, shading="auto", vmin=vmin, vmax=vmax)

    # --- Mark specific data point ---
    if highlight_index < len(df):
        r_pt = r_vals[highlight_index]
        d_pt = d_vals[highlight_index]
        ax.plot(r_pt, d_pt, "x", color="white", markersize=2, mew=0.8)

    # --- Minimal style ---
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(90)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(False)
    ax.set_frame_on(False)


# --- Subplots ---
fig, axes = plt.subplots(1, 2, figsize=figsize_inch, subplot_kw=dict(projection="polar"), dpi=600)

make_polar(axes[0], data1)
make_polar(axes[1], data2)

plt.subplots_adjust(wspace=0.02, hspace=0)
plt.show()
